# 08 — Analysis & Thesis Figures

Loads all evaluation results, builds comparison DataFrames, and generates
publication-ready figures for the thesis.

Conditions (each isolates a single degradation variable):
**clear_day** | **rainy_day** | **snowy_day** | **night_clear** | **overcast_day** | **partly_cloudy_day** | **dawn_dusk_clear**

Outputs are saved to `results/figures/`.

**Prerequisites:** notebooks 06 and 07 must have completed successfully.

In [ ]:
from pathlib import Path
import json
import pandas as pd

from src.eval_utils import build_comparison_df
from src.plot_utils import (
    plot_map_comparison,
    plot_degradation_curves,
    plot_efficiency_scatter,
    plot_per_class_heatmap,
)
from src.data_utils import CLASS_NAMES

DRIVE_ROOT   = Path('/content/drive/MyDrive/FON/master_rad/bdd100k_data')
RESULTS_ROOT = DRIVE_ROOT / 'results'
FIGURES_DIR  = RESULTS_ROOT / 'figures'
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

MODELS = ['yolov11', 'yolov12', 'rtdetr', 'rfdetr']
ALL_CONDITIONS = [
    'clear_day',
    'rainy_day', 'snowy_day', 'night_clear',
    'overcast_day', 'partly_cloudy_day', 'dawn_dusk_clear',
]
ADVERSE_CONDITIONS = ALL_CONDITIONS[1:]

In [ ]:
all_scores = {}
for model in MODELS:
    all_scores[model] = {}

for condition in ALL_CONDITIONS:
    scores_path = RESULTS_ROOT / condition / 'scores.json'
    if not scores_path.exists():
        print(f'WARNING: {scores_path} not found — skipping')
        continue
    scores = json.loads(scores_path.read_text())
    for model, vals in scores.items():
        all_scores.setdefault(model, {})[condition] = vals

df = build_comparison_df(all_scores)
print(df.to_string(index=False))

In [ ]:
# mAP@50 comparison bar chart (all models × all conditions)
fig = plot_map_comparison(df, metric='map50', output_path=FIGURES_DIR / 'map50_comparison.png')
fig.show()

In [ ]:
# mAP@50-95 comparison bar chart
fig = plot_map_comparison(df, metric='map50_95', output_path=FIGURES_DIR / 'map50_95_comparison.png')
fig.show()

In [ ]:
# Robustness degradation curves
fig = plot_degradation_curves(df, output_path=FIGURES_DIR / 'degradation_curves.png')
fig.show()

In [ ]:
hw_df = pd.read_json(str(RESULTS_ROOT / 'clear_day' / 'hw_metrics.json')).T.reset_index().rename(columns={'index': 'model'})

for condition in ADVERSE_CONDITIONS:
    fig = plot_efficiency_scatter(
        df, hw_df,
        x_metric='gflops',
        y_metric='retention_pct',
        condition=condition,
        output_path=FIGURES_DIR / f'efficiency_vs_robustness_{condition}.png',
    )
    fig.show()

In [ ]:
# TODO: Per-class AP heatmap — requires per_class_ap dicts from eval notebooks.
# Uncomment once compute_per_class_ap() is implemented:
#
# per_class_data = { ... }   # {model: {condition: {class_name: ap}}}
# for condition in ALL_CONDITIONS:
#     fig = plot_per_class_heatmap(
#         per_class_data, CLASS_NAMES, condition=condition,
#         output_path=FIGURES_DIR / f'per_class_ap_{condition}.png',
#     )
#     fig.show()

In [ ]:
csv_path = RESULTS_ROOT / 'comparison_table.csv'
df.to_csv(csv_path, index=False)
print('Comparison table saved to', csv_path)
print('Figures saved to', FIGURES_DIR)